# 日次動画統計の収集

7チャンネル分の全動画統計（再生数・いいね・コメント数）を取得してCSV保存する。
**毎日実行する唯一のノートブック。**

- 保存先（正）: `MyDrive/sixfonia_yt_analytics/<channel>/`
- 移行期のみ `MyDrive/YouTube_Data/` にもデュアルライト（`config.LEGACY_WRITE` で制御）
- スキーマは最初から統一済み: `videoId, viewCount, likeCount, commentCount, videoURL, view_date`

事前準備: Colab Secrets(🔑) に `YOUTUBE_API_KEY` と `GITHUB_TOKEN` を登録

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ▶️ 全チャンネル日次収集を実行
from sixfonia_analytics import auth, collect

youtube = auth.build_youtube()
print("収集日:", collect.today_str())

collect.collect_all_channels(youtube)

In [ ]:
#@title 📋 本日保存されたファイルの確認
import pandas as pd
from sixfonia_analytics import config, collect

date_str = collect.today_str()
rows = []
for name in config.CHANNEL_NAMES:
    for folder in [config.channel_dir(name), config.LEGACY_DATA_DIR]:
        p = folder / config.stats_filename(name, date_str)
        rows.append({
            "channel": name,
            "folder": folder.name,
            "exists": p.exists(),
            "size_kb": round(p.stat().st_size / 1024, 1) if p.exists() else None,
        })
pd.DataFrame(rows)

## 移行メモ

分析ノートブックがすべて `sixfonia_yt_analytics/` 参照に切り替わったら、
リポジトリの `sixfonia_analytics/config.py` の `LEGACY_WRITE = False` に変更して
`YouTube_Data/` への書き込みを停止する。